In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

from datetime import datetime, timezone
import json

# Workflow settings
# Set *_enabled = False to skip a step (e.g. when re-running only validation)
WORKFLOW_CONFIG = {
    "sync_enabled":         True,   # OneDrive → Lakehouse PDF sync
    "parse_enabled":        True,   # Parse PDFs → Delta tables
    "semantic_enabled":     True,   # Rebuild sem_* gold tables
    "validation_enabled":   True,   # Data-quality checks
    "notification_enabled": False,
    "notification_email":   "admin@example.com",
}

# Pre-initialise step outcome flags so summary always has them in scope
sync_success     = not WORKFLOW_CONFIG["sync_enabled"]
parse_success    = not WORKFLOW_CONFIG["parse_enabled"]
semantic_success = not WORKFLOW_CONFIG["semantic_enabled"]

# OneDrive sync settings
ONEDRIVE_CONFIG = {
    "client_id": "55636aca-83c5-4e15-ab25-8df679261286",
    "tenant_id": "0b60fed4-5fc9-409d-95f2-271114f4c86f",
    "scopes": ["https://graph.microsoft.com/.default"],
    "sharing_link": "https://relianceinfo-my.sharepoint.com/:f:/g/personal/kingsley_relianceinfosystems_com/IgBEoymEG5W2S7cdEVa9NCK9Aa2r55L84O8nTpYQvjKi3Ek?e=GhzdcW",
    "lakehouse_base_path": "Files",
}

print("✅ Configuration loaded")
print(f"   Sync enabled:       {WORKFLOW_CONFIG['sync_enabled']}")
print(f"   Parse enabled:      {WORKFLOW_CONFIG['parse_enabled']}")
print(f"   Semantic enabled:   {WORKFLOW_CONFIG['semantic_enabled']}")
print(f"   Validation enabled: {WORKFLOW_CONFIG['validation_enabled']}")


StatementMeta(, bae3fab5-bc65-4142-b76f-26b9ece1ab10, 7, Finished, Available, Finished, False)

✅ Configuration loaded
   Sync enabled:       False
   Parse enabled:      False
   Semantic enabled:   False
   Validation enabled: True


In [2]:
# ============================================================================
# STEP 1: SYNC PDFs FROM ONEDRIVE TO LAKEHOUSE
# ============================================================================

if WORKFLOW_CONFIG["sync_enabled"]:
    print("\n" + "=" * 80)
    print("STEP 1: SYNCING PDFs FROM ONEDRIVE")
    print("=" * 80 + "\n")

    # Delegate to the dedicated OneDrive_Fabric_Sync notebook which handles
    # its own auth (client-credentials / cached token) — avoids interactive
    # acquire_token_interactive() which is unsupported in Fabric Spark sessions.
    try:
        notebookutils.notebook.run(
            "OneDrive_Fabric_Sync",
            timeoutSeconds=3600,
            arguments={},
        )
        print("✅ OneDrive sync completed successfully")
        sync_success = True
    except Exception as e:
        print(f"❌ OneDrive sync failed: {e}")
        sync_success = False
else:
    print("⏭️ Sync step skipped (disabled in config)")
    sync_success = True


StatementMeta(, bae3fab5-bc65-4142-b76f-26b9ece1ab10, 8, Finished, Available, Finished, False)

⏭️ Sync step skipped (disabled in config)


In [ ]:
# ============================================================================
# STEP 2: PARSE PDFs AND INGEST INTO DELTA TABLES
# ============================================================================

if WORKFLOW_CONFIG["parse_enabled"] and sync_success:
    print("\n" + "=" * 80)
    print("STEP 2: PARSING PDFs AND INGESTING DATA")
    print("=" * 80 + "\n")

    try:
        result = notebookutils.notebook.run(
            "parse_assessment_pdfs",
            timeoutSeconds=3600,
            arguments={}
        )
        print("✅ PDF parsing completed successfully")
        parse_success = True
    except Exception as e:
        print(f"❌ PDF parsing failed: {e}")
        parse_success = False
else:
    if not WORKFLOW_CONFIG["parse_enabled"]:
        print("⏭️ Parse step skipped (disabled in config)")
    elif not sync_success:
        print("⏭️ Parse step skipped (sync failed)")
    parse_success = False


StatementMeta(, bae3fab5-bc65-4142-b76f-26b9ece1ab10, 9, Finished, Available, Finished, False)

⏭️ Parse step skipped (disabled in config)


In [ ]:
# ============================================================================
# STEP 2.5: REBUILD SEMANTIC GOLD LAYER
# ============================================================================

if WORKFLOW_CONFIG["semantic_enabled"] and parse_success:
    print("\n" + "=" * 80)
    print("STEP 2.5: REBUILDING SEMANTIC GOLD LAYER")
    print("=" * 80 + "\n")

    try:
        result = notebookutils.notebook.run(
            "build_semantic_layer",
            timeoutSeconds=3600,
            arguments={}
        )
        print("✅ Semantic layer rebuilt successfully")
        semantic_success = True
    except Exception as e:
        print(f"⚠️ Semantic layer rebuild failed: {e}")
        semantic_success = False
elif not WORKFLOW_CONFIG["semantic_enabled"]:
    print("⏭️ Semantic layer step skipped (disabled in config)")
    semantic_success = True
else:

    print("⏭️ Semantic layer step skipped (parse failed)")
    semantic_success = False

StatementMeta(, bae3fab5-bc65-4142-b76f-26b9ece1ab10, 10, Finished, Available, Finished, False)

⏭️ Semantic layer step skipped (disabled in config)


In [5]:
# ============================================================================
# STEP 3: VALIDATE DATA QUALITY
# ============================================================================

# Run validation if explicitly enabled AND either parse ran OK or parse was skipped intentionally
_validation_prereq = parse_success or not WORKFLOW_CONFIG["parse_enabled"]
if WORKFLOW_CONFIG["validation_enabled"] and _validation_prereq:
    print("\n" + "=" * 80)
    print("STEP 3: VALIDATING DATA QUALITY")
    print("=" * 80 + "\n")

    validation_results = {}

    checks = [
        ("security_assessment_assessments", "tenant_name", "overall_score_pct"),
        ("copilot_readiness_assessments",   "tenant_name", "overall_score_pct"),
        ("copilot_assessment_assessments",  "tenant_name", "overall_score_pct"),
        ("dbo.sem_fact_check",              "tenant_name", "is_latest_assessment"),
        ("dbo.sem_fact_posture",            "customer_key", "latest_readiness_score"),
        ("dbo.sem_fact_recommendation",     "customer_key", "matched_issue_count"),
    ]

    for table, key_col, score_col in checks:
        try:
            result = spark.sql(f"""
                SELECT COUNT(*) as total, COUNT({key_col}) as has_key
                FROM {table}
            """).collect()[0]
            completeness = (result.has_key / result.total * 100) if result.total > 0 else 0
            validation_results[table] = {"total": result.total, "completeness": completeness}
            print(f"✓ {table:<45} {result.total:>7,} rows  ({completeness:.1f}% complete)")
        except Exception as e:
            print(f"✗ {table}: {e}")
            validation_results[table] = {"error": str(e)}

    validation_success = True
else:
    print("⏭️ Validation step skipped")
    validation_success = False

    validation_results = {}

StatementMeta(, bae3fab5-bc65-4142-b76f-26b9ece1ab10, 11, Finished, Available, Finished, False)


STEP 3: VALIDATING DATA QUALITY

✓ security_assessment_assessments                   174 rows  (100.0% complete)
✓ copilot_readiness_assessments                     185 rows  (100.0% complete)
✓ copilot_assessment_assessments                     14 rows  (100.0% complete)
✓ dbo.sem_fact_check                             20,628 rows  (100.0% complete)
✓ dbo.sem_fact_posture                              353 rows  (100.0% complete)
✓ dbo.sem_fact_recommendation                     1,585 rows  (100.0% complete)


In [6]:
# ============================================================================
# STEP 4: WORKFLOW SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("WORKFLOW SUMMARY")
print("=" * 80 + "\n")

workflow_end = datetime.now(timezone.utc)

# Status relative to which steps were actually enabled
_enabled_steps_ok = (
    (not WORKFLOW_CONFIG["sync_enabled"]     or sync_success) and
    (not WORKFLOW_CONFIG["parse_enabled"]    or parse_success) and
    (not WORKFLOW_CONFIG["semantic_enabled"] or semantic_success)
)
all_ok = _enabled_steps_ok
workflow_status = "SUCCESS" if all_ok else ("PARTIAL" if (sync_success or parse_success) else "FAILED")

summary = {
    "workflow_status": workflow_status,
    "timestamp": workflow_end.isoformat(),
    "steps": {
        "1_sync":       ("✅ Success" if sync_success     else "❌ Failed") if WORKFLOW_CONFIG["sync_enabled"]     else "⏭️ Skipped",
        "2_parse":      ("✅ Success" if parse_success    else "❌ Failed") if WORKFLOW_CONFIG["parse_enabled"]    else "⏭️ Skipped",
        "2.5_semantic": ("✅ Success" if semantic_success else "❌ Failed") if WORKFLOW_CONFIG["semantic_enabled"] else "⏭️ Skipped",
        "3_validation": "✅ Success" if validation_success else "⏭️ Skipped",
    },
    "validation_results": validation_results,
}

print(f"Overall status: {workflow_status}")
print(f"Completed at:   {workflow_end.strftime('%Y-%m-%d %H:%M:%S UTC')}\n")
for step, status in summary["steps"].items():
    print(f"  {step}: {status}")

if validation_results:
    print("\nData Quality:")
    for table, metrics in validation_results.items():
        if "error" not in metrics:
            print(f"  {table}: {metrics['total']:,} rows ({metrics['completeness']:.1f}% complete)")

try:
    log_dir = "/lakehouse/default/Files/workflow_logs"
    import os; os.makedirs(log_dir, exist_ok=True)
    summary_file = f"{log_dir}/{workflow_end.strftime('%Y-%m-%d_%H-%M-%S')}_summary.json"
    with open(summary_file, "w") as fh:
        fh.write(json.dumps(summary, indent=2))
    print(f"\n📁 Summary saved: Files/workflow_logs/{workflow_end.strftime('%Y-%m-%d_%H-%M-%S')}_summary.json")
except Exception as e:
    print(f"\n⚠️ Could not save summary: {e}")

print("\n" + "=" * 80)
print(f"✅ WORKFLOW COMPLETE: {workflow_status}")
print("=" * 80)


StatementMeta(, bae3fab5-bc65-4142-b76f-26b9ece1ab10, 12, Finished, Available, Finished, False)


WORKFLOW SUMMARY

Overall status: SUCCESS
Completed at:   2026-07-11 01:37:21 UTC

  1_sync: ⏭️ Skipped
  2_parse: ⏭️ Skipped
  2.5_semantic: ⏭️ Skipped
  3_validation: ✅ Success

Data Quality:
  security_assessment_assessments: 174 rows (100.0% complete)
  copilot_readiness_assessments: 185 rows (100.0% complete)
  copilot_assessment_assessments: 14 rows (100.0% complete)
  dbo.sem_fact_check: 20,628 rows (100.0% complete)
  dbo.sem_fact_posture: 353 rows (100.0% complete)
  dbo.sem_fact_recommendation: 1,585 rows (100.0% complete)

📁 Summary saved: Files/workflow_logs/2026-07-11_01-37-21_summary.json

✅ WORKFLOW COMPLETE: SUCCESS
